## 📘 进阶 Transformer 注意力机制：MHA, MQA, GQA 与 MLA 笔记

在 Transformer 推理时，自回归生成需要缓存历史的 Key 和 Value（即 **KV Cache**）。随着上下文变长，KV Cache 的显存占用会呈线性爆炸式增长。这四种技术的发展，本质上就是一部**与 KV Cache 显存占用作斗争的演进史**。

### 1. MHA (Multi-Head Attention) - 标准多头注意力
* **核心思想**：每个 Query 头都有一个专属的 Key 头和 Value 头。
* **痛点**：KV Cache 极其庞大。$H$ 个头需要缓存 $H$ 份 KV 张量。

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class MHA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Q, K, V 投影维度相同：d_model -> d_model
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        B, S, D = x.shape
        
        # 1. 线性投影并重塑形状为: [B, S, num_heads, head_dim]
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(B, S, self.num_heads, self.head_dim)
        v = self.v_proj(x).view(B, S, self.num_heads, self.head_dim)
        
        # 2. 调换维度以计算注意力: [B, num_heads, S, head_dim]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # 3. 标准注意力计算 (内置了 scale 和 softmax)
        out = F.scaled_dot_product_attention(q, k, v)
        
        # 4. 拼接多头并输出: [B, S, d_model]
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return out
```

---

### 2. MQA (Multi-Query Attention) - 多查询注意力
* **核心思想**：为了极致压缩显存，**所有的 Query 头共享仅仅 1 个 Key 头和 1 个 Value 头**。
* **优势**：KV Cache 缩小到 MHA 的 $\frac{1}{H}$。
* **缺点**：大幅削弱了模型的表达能力，可能导致性能下降（参数量和非线性能力骤减）。

```python
class MQA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        # 注意！K 和 V 的输出维度仅仅是 1 个 head_dim
        self.k_proj = nn.Linear(d_model, self.head_dim)
        self.v_proj = nn.Linear(d_model, self.head_dim)

    def forward(self, x):
        B, S, D = x.shape
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        # K, V 形状: [B, S, 1, head_dim] -> [B, 1, S, head_dim]
        k = self.k_proj(x).view(B, S, 1, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, 1, self.head_dim).transpose(1, 2)
        
        # MQA 中，K 和 V 会利用 PyTorch 的广播机制 (Broadcasting) 
        # 自动扩展到与 Q 相同的 num_heads 维度进行计算
        out = F.scaled_dot_product_attention(q, k, v)
        
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return out
```

---

### 3. GQA (Grouped-Query Attention) - 分组查询注意力
* **核心思想**：MHA 和 MQA 的完美折中。将 Query 头分为 $G$ 组，**每组 Query 共享 1 个 Key 头和 1 个 Value 头**。
* **优势**：KV Cache 大幅减小（通常是 MHA 的 $\frac{1}{4}$ 或 $\frac{1}{8}$），但保留了接近 MHA 的精度，是 Llama 2/3、Mistral 等主流大模型的标配。

```python
class GQA(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads # 例如 num_heads=32, num_kv_heads=8
        self.num_queries_per_kv = num_heads // num_kv_heads # 每组 4 个 Q 共享 1 个 KV
        self.head_dim = d_model // num_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        # K 和 V 的维度对应 num_kv_heads
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim)

    def forward(self, x):
        B, S, D = x.shape
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 形状: [B, num_kv_heads, S, head_dim]
        k = self.k_proj(x).view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # 将 K, V 在 head 维度复制展开，以匹配 Q 的 num_heads
        # 形状变为: [B, num_heads, S, head_dim]
        k = torch.repeat_interleave(k, repeats=self.num_queries_per_kv, dim=1)
        v = torch.repeat_interleave(v, repeats=self.num_queries_per_kv, dim=1)
        
        out = F.scaled_dot_product_attention(q, k, v)
        
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return out
```

---

### 4. MLA (Multi-Head Latent Attention) - 多头潜变量注意力
* **核心思想**：由 DeepSeek 提出。不直接缓存庞大的多头 K 和 V 张量，而是通过**低秩投影 (Low-Rank Projection)** 将 KV 压缩成一个极小的“潜变量 (Latent Vector)”。
* **解耦 RoPE (Decoupled RoPE)**：因为低秩压缩会破坏旋转位置编码（RoPE）的结构，MLA 巧妙地分离出了一小部分 Query 和 Key 专门用来携带位置信息，这部分不参与压缩。
* **优势**：缓存量比 MQA 还小，但通过动态上采样（Up-projection）恢复多头维度，精度甚至超越 MHA。



```python
class MLA(nn.Module):
    def __init__(self, d_model, num_heads, kv_lora_rank, qk_rope_head_dim):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # --- KV 的低秩压缩与还原 ---
        # 1. 压缩层：将输入映射为低维潜变量 c_kv。推理时【只缓存】这个潜变量！
        self.kv_a_proj = nn.Linear(d_model, kv_lora_rank) 
        
        # 2. 还原层：从潜变量恢复出完整的 K 和 V (维度: num_heads * head_dim)
        self.kv_b_proj = nn.Linear(kv_lora_rank, self.num_heads * self.head_dim * 2)
        
        # --- Q 的处理 ---
        self.q_proj = nn.Linear(d_model, self.num_heads * self.head_dim)
        
        # --- 解耦的 RoPE 投影 ---
        # 专门分出一部分维度用于旋转位置编码，这部分不参与低秩压缩
        self.q_rope_proj = nn.Linear(d_model, self.num_heads * qk_rope_head_dim)
        self.k_rope_proj = nn.Linear(d_model, qk_rope_head_dim) # K 的 RoPE 共享一个头即可

    def forward(self, x):
        B, S, D = x.shape
        
        # 1. 计算 Q 内容
        q_c = self.q_proj(x).view(B, S, self.num_heads, self.head_dim)
        
        # 2. 计算 KV 潜变量并还原 (核心：低秩矩阵分解思想)
        c_kv = self.kv_a_proj(x) # 形状: [B, S, kv_lora_rank] -> 实际推理时存在这里即可
        
        # 动态还原出全量的 K 和 V
        kv_reconstructed = self.kv_b_proj(c_kv) 
        k_c, v_c = kv_reconstructed.chunk(2, dim=-1)
        k_c = k_c.view(B, S, self.num_heads, self.head_dim)
        v_c = v_c.view(B, S, self.num_heads, self.head_dim)
        
        # 3. 处理解耦的 RoPE 位置编码 (这里省略了应用 RoPE 的函数细节)
        q_r = self.q_rope_proj(x).view(B, S, self.num_heads, -1)
        k_r = self.k_rope_proj(x).view(B, S, 1, -1)
        
        # 4. 拼接内容与位置信息
        # Q = 拼接(Q_content, Q_rope)
        # K = 拼接(K_content, K_rope)
        q = torch.cat([q_c, q_r], dim=-1).transpose(1, 2)
        k = torch.cat([k_c, k_r.expand(-1, -1, self.num_heads, -1)], dim=-1).transpose(1, 2)
        v = v_c.transpose(1, 2)
        
        # 5. 标准注意力计算
        out = F.scaled_dot_product_attention(q, k, v)
        
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        # 最后通常还会过一个 output 投影层映射回 d_model
        return out
```

---

### 5. 全面性能与内存对比

假设模型参数：$H=32$ (总头数), $D_h=128$ (单头维度), $L$ (序列长度)。

| 特性 | MHA | MQA | GQA | MLA |
| :--- | :--- | :--- | :--- | :--- |
| **独立 Key/Value 头数** | $32$ | $1$ | 例如 $8$ | 不存完整的头，存潜变量 |
| **单层每个 Token KV Cache 占用** | $2 \times 32 \times 128 = 8192$ | $2 \times 1 \times 128 = 256$ | $2 \times 8 \times 128 = 2048$ | 取决于 rank，如 $512$ 潜维度 + $64$ RoPE 维度 = $576$ |
| **显存节约比例** | 基准 ($1\times$) | 极致压缩 ($32\times$) | 良好平衡 ($4\times$) | **理论极限 ($\sim14\times$)** |
| **模型表达能力 / 效果** | 最强 (基准) | 明显下降 | 几乎无损 | **等效于全头，极少损失** |
| **代表应用场景** | GPT-3, Transformer 经典结构 | StarCoder, 早期资源受限模型 | Llama 2/3, Qwen 2, 行业主流 | **DeepSeek-V2/V3，超长上下文与极致吞吐** |

---

这份笔记中的代码已经展示了 $Q, K, V$ 张量是如何在不同维度下生成、变换以及交互的。

关于这四种机制，你需要我针对**具体哪一种（比如 MLA 潜变量的矩阵乘法优化细节）**进一步展开，或者需要我演示如何把这段代码放进一个真实的 Decoder 块中测试吗？